In [ ]:
from haystack import Document
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.embedders import SentenceTransformersDocumentEmbedder

document_store = InMemoryDocumentStore()

# Adjust the Path if needed
file_paths = [
    "passenger_check_in_counter_all.txt", 
    "passenger_check_in_machine_all.txt", 
    "Overoccupied_flights_all.txt", 
    "delayed_luggage_All.txt"
]
#logs
print("Docs found successfully")

docs = []
for path in file_paths:
    # path as raw text if needed
    with open(path, "r", encoding="utf-8") as f:
        content = f.read()
        docs.append(Document(content=content, meta={"name": path}))

doc_embedder = SentenceTransformersDocumentEmbedder(model="sentence-transformers/all-MiniLM-L6-v2")
doc_embedder.warm_up()

docs_with_embeddings = doc_embedder.run(documents=docs)["documents"]


document_store.write_documents(docs_with_embeddings)
#logs
print(f"Successfully added {document_store.count_documents()} embedded documents to the store.")

In [ ]:
from haystack.components.builders import ChatPromptBuilder, PromptBuilder
from haystack_integrations.components.generators.ollama import OllamaGenerator
from haystack.dataclasses import ChatMessage
from haystack import component
from haystack import Pipeline

In [ ]:
# 1st template, user-system 
chat_template = [
    ChatMessage.from_system("You are a BPMN expert. Your task is to describe process models clearly."),
    ChatMessage.from_user("Describe the following BPMN model in detail: {{ documents[0].content }}")
]
chat_prompt_builder = ChatPromptBuilder(template=chat_template)

#2nd Template, with example
few_shot_template = """
You are a technical and articulate writer. Convert the BPMN XML into a descriptive and fully understandable text.

Example Input:
<bpmn:task id="Activity_1" name="Login"> ...

Example Output:
The user performs the Login task.

Main Task:
Input: {{ documents[0].content }}
Output:
"""
few_shot_prompt_builder = PromptBuilder(template=few_shot_template)

# 3rd template, without example
zero_shot_template = """
Read the following BPMN XML file and write a textual description of the process.
BPMN File:
{{ documents[0].content }}

Description:
"""
zero_shot_prompt_builder = PromptBuilder(template=zero_shot_template)

In [ ]:
ollama_config = {
    "model": "llama3.1:8b",
    "url": "http://localhost:11434",
    "timeout": 500,
    "generation_kwargs": {
        "num_ctx": 4096,
        "temperature": 0.9
    }
}

generator_1 = OllamaGenerator(**ollama_config)
generator_2 = OllamaGenerator(**ollama_config)
generator_3 = OllamaGenerator(**ollama_config)

In [ ]:
@component
class ResultMerger:
    @component.output_types(plain_texts=list[str])
    def run(self, reply_1: list[ChatMessage], reply_2: list[str], reply_3: list[str]):
        if reply_1:
            text_1 = reply_1[0].content
        else:
            text_1 = "No response"

        text_2 = reply_2[0] if reply_2 else "No response"
        text_3 = reply_3[0] if reply_3 else "No response"
        
        return {"plain_texts": [text_1, text_2, text_3]}

merger = ResultMerger()

In [ ]:
import re

def remove_coordinates(xml_data):
    #Remove the </bpmndi:BPMNDiagram> part
    return re.sub(r'<bpmndi:BPMNDiagram.*?</bpmndi:BPMNDiagram>','', xml_data, flags=re.DOTALL).strip()
#Test the function on a file that has not been touched up on manually

filename = "passenger_security.bpmn"

try:
    with open(filename, "r", encoding="utf-8") as f:
        content = f.read()
    print(f"File '{filename}' loaded.")

    if "bpmndi:BPMNDiagram" in content:
        print("The file has coordinates.")
    else:
        print("The file is already clean.")

    clean_content = remove_coordinates(content)

    if "bpmndi:BPMNDiagram" not in clean_content:
        print("The coordinates have been succesfully removed.")
        print(f"Characters before: {len(content)},  Characters after: {len(clean_content)}")
    else:
        print("The coordinates are still there.")

except FileNotFoundError:
    print(f"The file '{filename}' has not been found")

In [ ]:
from haystack import Pipeline
from haystack.components.embedders import SentenceTransformersTextEmbedder
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever
from haystack_integrations.components.generators.ollama import OllamaChatGenerator

#We need a text embedder and a retriever
text_embedder = SentenceTransformersTextEmbedder(model="sentence-transformers/all-MiniLM-L6-v2")
retriever = InMemoryEmbeddingRetriever(document_store=document_store)

#Create Pipeline
pipeline = Pipeline()

#Note: In order to get the response from bellow we have commented out the lines that include generators 1 and 3, as we would get a timeout error as we tried to run the 3 generators sat the same time
generator_1 = OllamaChatGenerator(**ollama_config) #we need the chat version for the first generator

#Add defined components
pipeline.add_component("text_embedder", text_embedder)
pipeline.add_component("retriever", retriever)

pipeline.add_component("chat_prompt_builder", chat_prompt_builder)
pipeline.add_component("few_shot_prompt_builder", few_shot_prompt_builder)
pipeline.add_component("zero_shot_prompt_builder", zero_shot_prompt_builder)

pipeline.add_component("llm_1", generator_1)
pipeline.add_component("llm_2", generator_2)
pipeline.add_component("llm_3", generator_3)

pipeline.add_component("merger", merger)

#Connect components
pipeline.connect("text_embedder.embedding", "retriever.query_embedding")

#Give the documents to all prompt builders
pipeline.connect("retriever.documents", "chat_prompt_builder.documents")
pipeline.connect("retriever.documents", "few_shot_prompt_builder.documents")
pipeline.connect("retriever.documents", "zero_shot_prompt_builder.documents")

#Send prompts to the generators
pipeline.connect("chat_prompt_builder.prompt", "llm_1.messages") 
pipeline.connect("few_shot_prompt_builder.prompt", "llm_2.prompt")
pipeline.connect("zero_shot_prompt_builder.prompt", "llm_3.prompt")

#Collect responses
pipeline.connect("llm_1", "merger.reply_1")
pipeline.connect("llm_2", "merger.reply_2")
pipeline.connect("llm_3", "merger.reply_3")

#Test
topics = [
    "Describe the process for delayed baggage.",
    "Describe the process for overoccupied flights."
]
for topic in topics:
    print(f"\n{'='*50}\nSubject: {topic}\n{'='*50}")
    
    result = pipeline.run({
        "text_embedder": {"text": topic},
        "retriever": {"top_k": 2}
    })

    texts = result["merger"]["plain_texts"]
    #When only running the second generator we used this line instead of the one from above
    #texts = result["llm_2"]["replies"]
    
    print(f"\n--- Generator 1 (Role Prompting) ---\n{texts[0]}")
    #When only running the second generator we used texts[0] to print out the results
    print(f"\n--- Generator 2 (Few-Shot) ---\n{texts[1]}")
    print(f"\n--- Generator 3 (Zero-Shot) ---\n{texts[2]}")

